In [1]:
# pip install openai chromadb beautifulsoup4 requests python-dotenv

import getpass, json, os, re, time, statistics
from pathlib import Path
from openai import AzureOpenAI

# Reads .env if you have one, otherwise uses the defaults below, so the
# notebook runs either way. Copy .env.example to .env to override.
try:
    from dotenv import load_dotenv
    load_dotenv()
except ImportError:
    pass

API_VERSION = os.getenv("AZURE_OPENAI_API_VERSION", "2025-01-01-preview")
CHAT_BASE   = os.getenv("AZURE_CHAT_BASE",  "https://api-iw.azure-api.net/sig-shared-jpeast-increased")
EMBED_BASE  = os.getenv("AZURE_EMBED_BASE", "https://api-iw.azure-api.net/sig-embedding")

CHAT_DEPLOYMENT   = os.getenv("CHAT_DEPLOYMENT",   "gpt-4o-mini")
VISION_DEPLOYMENT = os.getenv("VISION_DEPLOYMENT", "gpt-5-mini")
EMBED_DEPLOYMENT  = os.getenv("EMBED_DEPLOYMENT",  "text-embedding-3-small")

# This gateway takes the FULL path as the endpoint: deployment, operation
# and api-version included. So a client is bound to one deployment, and
# we need three of them.
#
# The chat route has no /openai segment, the embedding route does. That
# asymmetry is real, so do not tidy them into matching.

CHAT_URL   = f"{CHAT_BASE}/deployments/{CHAT_DEPLOYMENT}/chat/completions?api-version={API_VERSION}"
VISION_URL = f"{CHAT_BASE}/deployments/{VISION_DEPLOYMENT}/chat/completions?api-version={API_VERSION}"
EMBED_URL  = f"{EMBED_BASE}/openai/deployments/{EMBED_DEPLOYMENT}/embeddings?api-version={API_VERSION}"

# .strip() matters: pasting into a prompt often picks up a trailing
# newline, and that alone produces a 401.
KEY = (os.getenv("AZURE_OPENAI_KEY")
       or getpass.getpass("Azure OpenAI key: ")).strip()


def _client(url):
    return AzureOpenAI(azure_endpoint=url, api_key=KEY, api_version=API_VERSION)


chat_client   = _client(CHAT_URL)
vision_client = _client(VISION_URL)
embed_client  = _client(EMBED_URL)

# Each is tested separately so one failure does not hide the others.
for label, fn in [
    ("chat  ", lambda: chat_client.chat.completions.create(
        model=CHAT_DEPLOYMENT, max_tokens=5,
        messages=[{"role": "user", "content": "Reply with one word: connected"}]
     ).choices[0].message.content.strip()),
    ("vision", lambda: vision_client.chat.completions.create(
        model=VISION_DEPLOYMENT,
        messages=[{"role": "user", "content": "Reply with one word: connected"}]
     ).choices[0].message.content.strip()),
    ("embed ", lambda: f"{len(embed_client.embeddings.create(model=EMBED_DEPLOYMENT, input=['test']).data[0].embedding)} dimensions"),
]:
    try:
        print(f"{label}  OK      {fn()}")
    except Exception as e:
        print(f"{label}  FAILED  {type(e).__name__}: {str(e)[:110]}")

# 401 means the path is right and the key is wrong.
# 404 means the path is wrong, not the key.

chat    OK      Linked
vision  OK      connected
embed   OK      1536 dimensions


In [2]:
# ---- helpers ------------------------------------------------------
import sys
from pathlib import Path

ROOT = Path.cwd() if (Path.cwd() / "data" / "chroma").exists() else Path.cwd().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from bot.store import get_store, add_to_store

def chat(messages, model=None, temperature=0.0, max_tokens=512):
    r = chat_client.chat.completions.create(
        model=model or CHAT_DEPLOYMENT, messages=messages,
        temperature=temperature, max_tokens=max_tokens)
    return r.choices[0].message.content or ""


def ask(prompt, system=None, **kw):
    msgs = ([{"role": "system", "content": system}] if system else [])
    return chat(msgs + [{"role": "user", "content": prompt}], **kw)


def embed(texts, batch_size=256):
    """Embed a LIST of strings. One call per string is the slow mistake:
    3,000 round trips at ~200ms each is ten minutes of network wait."""
    texts = [t.replace("\n", " ") for t in texts]
    out = []
    for i in range(0, len(texts), batch_size):
        r = embed_client.embeddings.create(model=EMBED_DEPLOYMENT, input=texts[i:i + batch_size])
        out.extend(d.embedding for d in r.data)
    return out


def chunk(text, size=800, overlap=100):
    """Fixed-size chunks with overlap. Defaults to start from, not
    recommended values."""
    text = " ".join(text.split())
    step = size - overlap
    return [text[i:i + size] for i in range(0, len(text), step)
            if text[i:i + size].strip()]


# def get_store(path="data/chroma", name="workshop", reset=False):
#     import chromadb, shutil
#     from pathlib import Path as _P
#     if reset and _P(path).exists():
#         shutil.rmtree(path)          # cleaner than deleting the collection
#     try:
#         c = chromadb.PersistentClient(path=path)
#         return c.get_or_create_collection(name)
#     except KeyError as e:
#         raise RuntimeError(
#             f"chromadb cannot read the index at {path} ({e}). It was built by a "
#             f"different chromadb version. Delete that folder and rebuild, or "
#             f"install the pinned version from requirements.txt."
#         ) from None

# def get_store(path="data/chroma", name="workshop", reset=False):
#     import chromadb

#     try:
#         c = chromadb.PersistentClient(path=path)
#     except KeyError as e:
#         raise RuntimeError(
#             f"ChromaDB cannot read the index at '{path}' ({e}). "
#             f"It was likely built with a different chromadb version.\n"
#             f"Fix: Restart your Jupyter kernel, manually delete the '{path}' folder, "
#             f"and verify versions with 'pip show chromadb'."
#         ) from None

#     if reset:
#         try:
#             c.delete_collection(name)
#         except Exception:
#             # Safe to ignore if collection doesn't exist yet
#             pass
#     return c.get_or_create_collection(name)


# def add_to_store(store, texts, metadatas, ids=None, batch_size=128):
#     ids = ids or [f"c{i}" for i in range(len(texts))]
#     for i in range(0, len(texts), batch_size):
#         sl = slice(i, i + batch_size)
#         store.add(ids=ids[sl], documents=texts[sl],
#                   embeddings=embed(texts[sl]), metadatas=metadatas[sl])


# def query(store, question, k=5, where=None):
#     """The k nearest chunks. Chroma returns squared L2, so lower is closer."""
#     r = store.query(query_embeddings=embed([question]), n_results=k,
#                     where=where or None)
#     return [{"text": d, "metadata": m, "distance": dist}
#             for d, m, dist in zip(r["documents"][0], r["metadatas"][0],
#                                   r["distances"][0])]

def query(store, question, k=5, where=None):
    # Fetch 2x candidates to filter duplicates
    r = store.query(query_embeddings=embed([question]), n_results=k * 2, where=where or None)
    
    unique_chunks = []
    seen_texts = set()
    for d, m, dist in zip(r["documents"][0], r["metadatas"][0], r["distances"][0]):
        # Deduplicate using normalized text prefix
        norm = " ".join(d.split()[:40])
        if norm not in seen_texts:
            seen_texts.add(norm)
            unique_chunks.append({"text": d, "metadata": m, "distance": dist})
        if len(unique_chunks) >= k:
            break
            
    return unique_chunks


def show(chunks, chars=200):
    if not chunks:
        print("  (nothing returned)")
        return
    for c in chunks:
        print(f"  {c['distance']:.3f}  {c['metadata'].get('url', '?')}")
        print(f"         {c['text'][:chars].strip()}\n")


def timed(fn, *a, **kw):
    t0 = time.time()
    return fn(*a, **kw), time.time() - t0


def normalise(s):
    return re.sub(r"[^a-z0-9 ]", " ", (s or "").lower())


def answer_present(expected, chunks):
    """Was the expected answer anywhere in the retrieved text? Crude, and
    enough to tell a retrieval failure from a prompt failure."""
    hay  = normalise(" ".join(c["text"] for c in chunks))
    need = normalise(expected).strip()
    if need and need in hay:
        return True
    terms = [t for t in need.split() if len(t) > 3]
    return bool(terms) and sum(t in hay for t in terms) / len(terms) >= 0.8


def load_dev_set(path="dev_set.json"):
    return json.loads(Path(path).read_text())


def describe_image(image_path, prompt, model=None):
    """One image plus an instruction. There is deliberately no default
    prompt: writing it is the lab 4 exercise."""
    import base64, mimetypes
    p    = Path(image_path)
    mime = mimetypes.guess_type(p.name)[0] or "image/jpeg"
    b64  = base64.b64encode(p.read_bytes()).decode()
    r = vision_client.chat.completions.create(
        model=model or VISION_DEPLOYMENT, max_tokens=800,
        messages=[{"role": "user", "content": [
            {"type": "text", "text": prompt},
            {"type": "image_url", "image_url": {"url": f"data:{mime};base64,{b64}"}},
        ]}])
    return r.choices[0].message.content or ""

print("helpers loaded")

helpers loaded


In [ ]:
# SYSTEM_PROMPT = """You answer factual questions about the Tam Wing Fan Innovation Wing.

# Answer only from the context below. Where the context disagrees with what
# you think you know, the context is correct.

# Reply with the answer only. No explanation, no preambleand do NOT say 
# "The context does not specify" or "My best guess is". If the question
# asks how many, reply with a number only.

# If asked what 3D printing is also known as, state the broad manufacturing category.
# If the context does not contain the answer, give your best guess anyway.
# Never reply that you do not know."""

# SYSTEM_PROMPT = """You are a precise, factual question-answering assistant for an engineering centre.
# Answer directly from the provided context.

# How to interpret the context:
# 1. Table Data: HTML tables are formatted with "|" separating columns. When reading a row, carefully map each value back to its exact corresponding column header from the top of the table.
# 2. Timelines: If a question specifies a year, extract the fact for that exact year and ignore past/future data.
# 3. Specificity: Return the exact names or values requested. For broad technologies, use the industry category.

# Output format:
# - Return ONLY the exact short answer. No conversational filler.
# - If asked for a count or capacity, output digits only."""

SYSTEM_PROMPT = """You answer factual questions about the Tam Wing Fan Innovation Wing.
Answer directly from the provided context. Where the context disagrees with what
you think you know, the context is correct.

Reply with the answer only. No explanation, no preamble and do NOT say  
"The context does not specify" or  "My best guess is".
If the context does not contain the answer, give your best guess anyway. Never reply that you do not know.

How to interpret the context:
1. Table Data: HTML tables are formatted with "|" separating columns. When reading a row, carefully map each value back to its exact corresponding column header from the top of the table.
2. Timelines: If a question specifies a year, extract the fact for that exact year and ignore past/future data.
3. Specificity: Return the exact names or values requested. For broad technologies, use the industry category. 

Output format:
- Return ONLY the exact short answer. No conversational filler.
- If asked for a count or capacity, output digits only."""


CONFIG = {
    "k": 18,
    "temperature": 0.0,
    "max_tokens": 128,
}

# def answer_one(store, question, config):
#     chunks = query(store, question, k=config.get("k", 15))
#     context = "\n\n".join(f"[{c['metadata'].get('url','?')}]\n{c['text']}" for c in chunks)
#     reply = chat([
#         {"role": "system", "content": SYSTEM_PROMPT},
#         {"role": "user", "content": f"Context:\n{context}\n\nQuestion: {question}"},
#     ])
#     return reply.strip(), chunks

def answer_one(store, question, config):
    # Fetch k chunks from the database
    chunks = query(store, question, k=config.get("k", 18))
    
    context_blocks = []
    seen_texts = set()
    
    for c in chunks:
        body = c['text'].strip()
        
        # 1. Deduplicate: if we already added this exact text, skip it entirely
        if body in seen_texts:
            continue
        seen_texts.add(body)
        
        # 2. Extract both URL and Year from metadata
        src = c['metadata'].get('url', 'document')
        year = c['metadata'].get('year', 'Unknown')
        
        # 3. Format it so the LLM explicitly sees the Year
        doc_num = len(context_blocks) + 1
        context_blocks.append(f"[Document {doc_num} | Year: {year} | Source: {src}]\n{body}")
        
    context = "\n\n---\n\n".join(context_blocks)
    
    reply = chat([
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": f"Context:\n{context}\n\nQuestion: {question}\nAnswer:"},
    ])
    
    return reply.strip(), chunks

def check_match(expected, predicted):
    exp = normalise(expected).strip()
    pred = normalise(predicted).strip()
    if not exp or not pred:
        return False
    # Exact or substring match
    if exp in pred or pred in exp:
        return True
    # Word overlap match for spelling variations (e.g., modeling vs modelling)
    exp_words = set(exp.split())
    pred_words = set(pred.split())
    overlap = exp_words & pred_words
    return len(overlap) / max(len(exp_words), 1) >= 0.6

def run_dev_set(store, dev_items, config):
    rows = []
    for item in dev_items:
        pred, chunks = answer_one(store, item["question"], config)
        hit = answer_present(item["answer"], chunks)
        answer_correct = check_match(item["answer"], pred)
        rows.append({
            "id": item.get("id"),
            "level": item.get("level", 1),
            "question": item["question"],
            "expected": item["answer"],
            "predicted": pred,
            "chunks": chunks,
            "hit": hit,
            "correct": answer_correct,
        })
    return rows

def review(rows):
    total = len(rows)
    hits = sum(1 for r in rows if r["hit"])
    correct_count = sum(1 for r in rows if r["correct"])

    print(f"Retrieval Hit Rate: {hits}/{total} ({hits/total*100:.1f}%)")
    print(f"Answer Accuracy:    {correct_count}/{total} ({correct_count/total*100:.1f}%)\n")

    for idx, r in enumerate(rows, 1):
        hit_mark = "HIT" if r["hit"] else "MISS"
        status = "CORRECT" if r["correct"] else "INCORRECT"
        print(f"[{idx}/{total}] [{status} | {hit_mark}] Lv{r.get('level', '?')} Q: {r['question']}")
        print(f"       Expected:  {r['expected']}")
        print(f"       Predicted: {r['predicted']}\n")

In [4]:
pages = json.loads(Path("data/pages.json").read_text(encoding="utf-8"))

makerspace_pages = [p for p in pages if "makerspace a" in p["text"].lower()]
print("Pages mentioning Makerspace A:", len(makerspace_pages))
for p in makerspace_pages[:3]:
    print(" -", p["url"])
    if "40" in p["text"]:
        print("   -> Found '40' in this page!")

funding_pages = [p for p in pages if "funding-scheme" in p["url"] or "funding scheme" in p["text"].lower()]
print("\nPages mentioning Funding Scheme:", len(funding_pages))
for p in funding_pages[:3]:
    print(" -", p["url"])
    if "17 october" in p["text"].lower() or "october 2025" in p["text"].lower():
        print("   -> Found deadline date in this page!")

Pages mentioning Makerspace A: 45
 - https://innowings.engg.hku.hk/makerspace/
 - https://innowings.engg.hku.hk/tutor/
 - https://innowings.engg.hku.hk/recruitment-of-innovation-wing-tutor-2025-2026/

Pages mentioning Funding Scheme: 42
 - https://innowings.engg.hku.hk/innowing1/fundings/
 - https://innowings.engg.hku.hk/innowing1/sig/
 - https://innowings.engg.hku.hk/past-events/


In [32]:
import chromadb

dev = load_dev_set("dev_set.json")
store = get_store(reset=False)

print("Collection count:", store.count())

# q = "What is the capacity of Makerspace A?"
# chunks = query(store, q, k=10)
# print(f"Retrieved {len(chunks)} chunks for: '{q}'\n")
# for idx, c in enumerate(chunks):
#     print(f"--- Chunk {idx+1} | Dist: {c['distance']:.3f} | {c['metadata'].get('url')} ---")
#     print(c["text"][:300], "\n")

# Run benchmark
rows = run_dev_set(store, dev, CONFIG)
review(rows)

# Check Lv1 & Lv2 scores specifically
lv1_lv2 = [r for r in rows if r.get("level") in (1, 2)]
passed = sum(1 for r in lv1_lv2 if r["correct"])
print(f"Checkpoint 1 Retrieval: {passed}/{len(lv1_lv2)} on Lv1/Lv2")

Collection count: 7167
Retrieval Hit Rate: 9/15 (60.0%)
Answer Accuracy:    5/15 (33.3%)

[1/15] [CORRECT | MISS] Lv1 Q: What is 3D printing also known as?
       Expected:  Additive manufacturing
       Predicted: Additive manufacturing

[2/15] [CORRECT | MISS] Lv1 Q: What does FDM stand for in 3D printing?
       Expected:  Fused deposition modelling
       Predicted: Fused Deposition Modeling

[3/15] [CORRECT | HIT] Lv1 Q: Which faculty would you expect an engineering innovation centre to belong to?
       Expected:  The Faculty of Engineering
       Predicted: Faculty of Engineering

[4/15] [INCORRECT | MISS] Lv2 Q: What is the capacity of Makerspace A?
       Expected:  40
       Predicted: 153

[5/15] [CORRECT | HIT] Lv2 Q: When was the deadline for the first round of the Funding Scheme in 2025?
       Expected:  17 October 2025
       Predicted: October 17, 2025

[6/15] [INCORRECT | HIT] Lv2 Q: Which two materials are banned from the laser cutter?
       Expected:  PVC and polyc

In [17]:
external_test_set = [
    {
        "level": 1,
        "question": "What does SIG stand for in the context of the Innovation Wing?",
        "expected": "Special Interest Group"
    },
    {
        "level": 2,
        "question": "What are the standard weekday opening hours for Tam Wing Fan Innovation Wing One?",
        "expected": "09:00 - 22:00" 
    },
    {
        "level": 2,
        "question": "Which software is used to prepare files for the GCC LaserPro Spirit laser cutter?",
        "expected": "CorelDRAW" 
    },
    {
        "level": 2,
        "question": "What is the capacity of Makerspace A?",
        "expected": "11" 
    }
]

print("--- Running External / Hold-Out Test Set ---\n")

for idx, item in enumerate(external_test_set, 1):
    q = item["question"]
    expected = item["expected"]
    
   
    pred, chunks = answer_one(store, q, CONFIG)
    
    print(f"[{idx}/{len(external_test_set)}] Lv{item['level']} Q: {q}")
    print(f"       Expected:  {expected}")
    print(f"       Predicted: {pred}")
    
    
    if chunks:
        top_url = chunks[0]['metadata'].get('url', 'Unknown')
        print(f"       Grounded on: {top_url}\n")
    else:
        print("       Grounded on: No chunks retrieved\n")

--- Running External / Hold-Out Test Set ---

[1/4] Lv1 Q: What does SIG stand for in the context of the Innovation Wing?
       Expected:  Special Interest Group
       Predicted: Student Interest Group
       Grounded on: https://innowings.engg.hku.hk/mwlko/

[2/4] Lv2 Q: What are the standard weekday opening hours for Tam Wing Fan Innovation Wing One?
       Expected:  09:00 - 22:00
       Predicted: 9:00am – 9:00pm
       Grounded on: https://innoacademy.engg.hku.hk/contact/

[3/4] Lv2 Q: Which software is used to prepare files for the GCC LaserPro Spirit laser cutter?
       Expected:  CorelDRAW
       Predicted: CorelDRAW
       Grounded on: https://innowings.engg.hku.hk/acrylic/

[4/4] Lv2 Q: What is the capacity of Makerspace A?
       Expected:  11
       Predicted: 60
       Grounded on: https://innowings.engg.hku.hk/makerspace/



In [7]:
pages = json.loads(Path("data/pages.json").read_text(encoding="utf-8"))
print("Pages mentioning Tidewatch:", [p["url"] for p in pages if "tidewatch" in p["text"].lower()])

Pages mentioning Tidewatch: []
